In [17]:
import sys

sys.path.insert(0, "src")

import travel_agent.config
from agents import Runner, trace

from travel_agent.agents.intake_agent import intake_agent
from travel_agent.agents.visa_agent import visa_agent
from travel_agent.agents.activities_agent import run_activities_agent
from travel_agent.agents.stay_agent import run_stay_agent

## 1. Intake: multi-turn chat


In [18]:
history = []

async def say(msg):
    global history
    result = await Runner.run(intake_agent, history + [{'role': 'user', 'content': msg}])
    history = result.to_input_list()
    out = result.final_output
    print('missing:', out.missing_fields)
    print('assistant:', out.next_question)
    return out

out = await say('I want to visit Japan with 2 friends next month for 7 days, budget 6000 USD')

missing: ['trip_type', 'passport_country', 'origin', 'start_date']
assistant: What is the purpose of your trip? Leisure or business?


In [19]:
out = await say('I hold a French passport, flying from Miami. Food and temples. Thinking of coming Wednesday')

missing: []
assistant: None


In [20]:
trip = out.trip
print(trip.model_dump_json(indent=2))
print('complete:', out.is_complete)

{
  "trip_type": "leisure",
  "passport_country": "France",
  "origin": "Miami",
  "destination": "Japan",
  "start_date": "2026-10-06",
  "end_date": "2026-10-12",
  "num_days": 7,
  "travelers": 3,
  "budget_usd": 6000.0,
  "interests": [
    "Food",
    "temples"
  ],
  "meeting_address": null,
  "needs_car_rental": null
}
complete: True


## 2. Visa agent


In [21]:
with trace(""):
    r = await Runner.run(visa_agent, f'Passport: {trip.passport_country}. Destination: {trip.destination}.')
print(r.final_output.model_dump_json(indent=2))

{
  "passport": "FRA",
  "destination": "JPN",
  "requirement": "visa_free",
  "allowed_days": 90,
  "summary": "French passport holders can enter Japan without a visa and stay visa-free for up to 90 days. This is a visa-free travel arrangement.",
  "source": "orizn_api",
  "disclaimer": "This information was last verified on 2026-05-08 and is provided for informational purposes only. For the most accurate and up-to-date details, please check with the official government sources of Japan."
}


## 3. Activities agent


In [16]:
prompt = (
    f'Destination: {trip.destination}. Dates: {trip.start_date} to {trip.end_date}. '
    f'Travelers: {trip.travelers}. Budget USD: {trip.budget_usd}. '
    f'Interests: {trip.interests}. Trip type: {trip.trip_type}.'
)
with trace('Day4 activities'):
    r = await Runner.run(activities_agent, prompt)
plan = r.final_output
print(plan.weather_summary)
for a in plan.activities:
    print('-', a.name, '|', a.category, '|', a.estimated_cost_usd)

The weather in Kyoto from 2026-10-06 to 2026-10-12 is generally warm with temperatures ranging from 12.6°C to 27.7°C. Rain probability is low for most days except for a 45% chance on the first day, so mostly outdoor activities are suitable.
- Kyoto Kare Seisakusho Kariru | Food | 30.0
- Nouveau Burger KYOTO | Food | 25.0
- Bungalow Terramachi | Food | 28.0
- Marumaru Seimen | Food | 20.0
- Izushi Handcrafted Soba Shop Deishisaku | Food | 22.0
- Portner City Memorial Tower | Temples & Sights | None
- Kyoto City Civic Memorial | Temples & Sights | None
- World Heritage Treaty 40th Anniversary Memorial | Temples & Sights | None
- Kyoto City Hall | Temples & Sights | None
- Kyoto Historical City Council & Declaration | Temples & Sights | None


In [22]:
with trace("Check Activities Agent"):
    activities = await run_activities_agent(trip)

print("Weather:")
print(activities.weather_summary)

print("\nActivities:")
for activity in activities.activities:
    print(
        f"- {activity.name} "
        f"| category: {activity.category} "
        f"| cost: ${activity.estimated_cost_usd} "
        f"| duration: {activity.duration_hours} hours "
        f"| from tool: {activity.from_tool}"
    )
    print(f"  Why: {activity.why}")

Weather:
The weather in Kyoto from October 6 to October 12, 2026, will generally be pleasant with daytime temperatures ranging from 22.6°C to 27.7°C and low probabilities of rain on most days, except for a 45% chance of rain on October 6. It is suitable for outdoor activities, especially from October 7 to October 12.

Activities:
- Kyoto Kare Seisakusho Kariru | category: Food | cost: $40.0 | duration: 1.5 hours | from tool: True
  Why: Recommended local restaurant in Kyoto area with good reviews.
- Nouvelle Vague Kyoto | category: Food | cost: $35.0 | duration: 1.5 hours | from tool: True
  Why: Popular dining spot specializing in local cuisine in central Kyoto.
- Bungalow Terramachi | category: Food | cost: $30.0 | duration: 1.5 hours | from tool: True
  Why: Well-known restaurant located in Teramachi Street area.
- Marumaru Seimen | category: Food | cost: $20.0 | duration: 1.0 hours | from tool: True
  Why: Renowned noodle shop famous in Kyoto for authentic taste.
- Izushi Tebuchi S

In [23]:
with trace("Check Stay Agent"):
    stay = await run_stay_agent(trip)

print("Stay notes:")
print(stay.notes)

print("\nAccommodation options:")
for option in stay.options:
    print(
        f"- {option.name} "
        f"| area: {option.area} "
        f"| total: ${option.total_cost_usd} "
        f"| estimate: {option.is_estimate}"
    )
    print(f"  Why: {option.why}")

Stay notes:
All prices are sample estimates, not actual booking prices. Final hotel cost depends on travel dates, number of rooms needed, applicable taxes, and availability.

Accommodation options:
- Sample Kyoto Guesthouse | area: Kyoto | total: $510.0 | estimate: True
  Why: Sample value option near traditional neighborhoods and temple areas.
- Sample Osaka Food District Hotel | area: Osaka | total: $660.0 | estimate: True
  Why: Sample option for food-focused travelers with easy access to city transport.
- Sample Tokyo City Hotel | area: Tokyo | total: $750.0 | estimate: True
  Why: Sample central option with convenient access to restaurants, transit, and major attractions.
